# LubriSet — Inference Notebook (Нефтекод 2026)

Запускает обученный ансамбль Set Transformer'ов и создаёт `predictions.csv` с двумя таргетами для каждого `scenario_id` из `daimler_mixtures_test.csv`.

Ожидает, что веса моделей лежат в папке `artifacts/` рядом с ноутбуком.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader

from src.data import (
    CONDITION_DIM, COL_COMP, TOP_PROPERTIES,
    build_component_property_table, build_scenario_samples, build_vocabs,
    load_properties, target_inverse_transform,
)
from src.model import LubriSet
from src.train import SetDataset, collate

DATA_DIR = Path('.')
ART_DIR = Path('artifacts')
DEVICE = torch.device('cpu')

In [ ]:
mix_train = pd.read_csv(DATA_DIR / 'daimler_mixtures_train.csv')
mix_test = pd.read_csv(DATA_DIR / 'daimler_mixtures_test.csv')
pr = load_properties(str(DATA_DIR / 'daimler_component_properties.csv'))
wide_batch, wide_comp, mu, sd = build_component_property_table(pr)
comp_vocab, type_vocab = build_vocabs(mix_train, mix_test)
train_comp_set = set(mix_train[COL_COMP].unique())
test_samples = build_scenario_samples(
    mix_test, wide_batch, wide_comp, mu, sd, comp_vocab, type_vocab,
    train_comp_set=train_comp_set, is_train=False,
)
print('Test scenarios:', len(test_samples))

In [ ]:
def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    cfg = ckpt['config']
    m = LubriSet(n_components=cfg['n_components'], n_types=cfg['n_types'],
                 n_props=cfg['n_props'], condition_dim=cfg['condition_dim'],
                 d_model=cfg['d_model'], n_layers=cfg['n_layers'], n_targets=2)
    m.load_state_dict(ckpt['state_dict'])
    m.to(DEVICE); m.eval()
    return m, ckpt

ckpts = sorted(ART_DIR.glob('model_fold*_seed*.pt'))
print(f'Found {len(ckpts)} checkpoints')

In [ ]:
all_preds = []
target_mu = target_sd = None
for ck in ckpts:
    model, meta = load_model(str(ck))
    if target_mu is None:
        target_mu = meta['target_mu']; target_sd = meta['target_sd']
    ds = SetDataset(test_samples)
    dl = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=collate)
    outs, ids = [], []
    with torch.no_grad():
        for batch in dl:
            batch_t = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}
            pred = model(batch_t['comp_ids'], batch_t['type_ids'], batch_t['props'], batch_t['miss'],
                         batch_t['mass'], batch_t['is_new'], batch_t['conditions'], batch_t['pad_mask'])
            outs.append(pred.cpu().numpy()); ids.extend(batch_t['scenario_id'])
    preds = np.concatenate(outs) * target_sd + target_mu
    all_preds.append(target_inverse_transform(preds))

avg = np.mean(np.stack(all_preds, axis=0), axis=0)
sub = pd.DataFrame({
    'scenario_id': ids,
    'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %': avg[:, 0],
    'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm': avg[:, 1],
})
sub.to_csv('predictions.csv', index=False)
print('Wrote predictions.csv with', len(sub), 'rows')
sub.head()